# KNN

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 2. HÀM ĐÁNH GIÁ (Lưu kết quả chung vào list)
results_list = []

def evaluate_model(model, version_name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    res = {
        'Version': version_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4)
    }

    results_list.append(res)
    print(f"[{version_name}] Acc: {res['Accuracy']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return model

### Danh sách lưu kết quả để xuất CSV

In [ ]:
results_list = []

def evaluate_model(model, name, X_test, y_test):
    # Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # Xác suất class 1 để tính AUC

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    res = {
        'Algorithm': name,
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-Score': round(f1, 2),
        'AUC': round(auc, 2)
    }

    results_list.append(res)
    print(f"[{name}] Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | F1: {f1:.2f} | AUC: {auc:.2f}")
    return res

### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [ ]:
print("--- V1: Baseline KNN ---")

# Baseline KNN
KNN_baseline_model = KNeighborsClassifier(n_neighbors=5)
KNN_baseline_model.fit(X_train, y_train)

evaluate_model(KNN_baseline_model, "V1: KNN Baseline", X_test, y_test)

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [ ]:
print("--- V2: GridSearchCV Tuning for KNN ---")

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['minkowski'],
    'p': [1, 2]  # p=1: Manhattan, p=2: Euclidean
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
KNN_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")
evaluate_model(KNN_tuned_model, "V2: KNN Tuned (GridSearch)", X_test, y_test)

### 5. Final KNN Model

In [ ]:
print("--- V3: Final Best KNN Model ---")

# Dùng lại mô hình tốt nhất từ GridSearchCV như mô hình cuối cùng
KNN_final_model = grid_search.best_estimator_
evaluate_model(KNN_final_model, "V3: Final KNN Model", X_test, y_test)

### (5) Lưu kết quả

In [ ]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/KNN/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU CSV
df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'KNN_evaluation_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'KNN_baseline.pkl'), 'wb') as f:
    pickle.dump(KNN_baseline_model, f)

with open(os.path.join(save_path, 'KNN_tuned.pkl'), 'wb') as f:
    pickle.dump(KNN_tuned_model, f)

with open(os.path.join(save_path, 'KNN_final.pkl'), 'wb') as f:
    pickle.dump(KNN_final_model, f)

print("-" * 40)
print(f"✅ Đã lưu file CSV tại: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)

In [ ]:
!jupyter nbconvert --to html KNN_model.ipynb